**Candidate number 26508**  
**This jupyter notebook should only be read after reading <font size="5">Wekipedia_webscraping.ipynb AND Further_Extracting.ipynb** </font>  
**Kernal: python 3.10.6**  
**Purpose: data analysis**  

#### 🤖In the section 2.13 in this jupyter notebook,  
#### I have gained the idea of using **datetime** module to process my editing time data by asking **Chatgpt**
#### 🤖In the section 2.222 in this jupyter notebook, I was suggested by AI to use **ast** to solve data format problems.

## Section 1 Basic preparations

Import modules needed

In [ ]:
import requests
from scrapy import Selector
import pandas as pd
from pprint import pprint
from tqdm.notebook import tqdm

Importing my files from specified path

In [ ]:
brief_data = r'D:\ds105a-2023-w07-summative-oche32\initial data\brief_wikipedia_data.csv'
time_data = r'D:\ds105a-2023-w07-summative-oche32\further data\all_time_data.csv'
stub_data = r'D:\ds105a-2023-w07-summative-oche32\further data\all_stubs_data.csv'

## Section 2 Data analysis

### **2.1: This block analyses data about the median of editing times in further_extraction.ipynb.** 

2.11 introducing a Python module that is helpful for time data analysis

In [ ]:
df_time = pd.read_csv(time_data)
from datetime import datetime
import csv
import re

2.12 :

The format of my all_time_data.csv contains too much excess information.  
I need to simplify "[' This page was last edited on xx xx 20xx, at 14:59', '\xa0(UTC)', '.']" to [xx xx 20xx]

In [ ]:
with open(time_data, 'r', encoding='utf-8') as input_file, open('simplified_time.csv', 'w', encoding='utf-8', newline='') as output_file:
    reader = csv.reader(input_file)
    writer = csv.writer(output_file)
    
    for row_index, row in enumerate(reader, start=1):
        if len(row) > 0:
            date_pattern = r'(\d+ \w+ \d{4})'  # "day month year" pattern
            match = re.search(date_pattern, row[0])
            if match:
                formatted_date = f"[{match.group(1)}]"
                writer.writerow([formatted_date])
            else:
                print(f"No valid date found in row {row_index}: {row}") # I want no invalid rows in the data.
print('simplification completed')

2.13:

To find the average last editing data, I need to find out the **Median** for **years, months, and dates**  

2.131 Initialize dictionaries to store counts for years, months, and dates

In [ ]:
year_counts = {}
month_counts = {month: 0 for month in range(1, 13)}
date_counts = {}

2.132 Process on the file to detect the number of days, months, and years.  
I defined a range for years because the sample size of years < 2017 is too small to have a median there.

In [ ]:
with open('simplified_time.csv', 'r', encoding='utf-8') as csvfile:
    csv_reader = csv.reader(csvfile)
    
    next(csv_reader)
    
    for row in csv_reader:
        element = row[0]
        try:
            date_obj = datetime.strptime(element, '[%d %B %Y]')
            if 2017 <= date_obj.year <= 2023:
                year_counts[str(date_obj.year)] = year_counts.get(str(date_obj.year), 0) + 1
                month_counts[date_obj.month] += 1
                
                day_part = date_obj.strftime('%d')
                date_counts[day_part] = date_counts.get(day_part, 0) + 1
        except ValueError:
            print(f"No valid date found in row {row}: {element}")


2.133 Print the counts for years, months, and dates within the specified range

In [ ]:
print("Year Counts:")
for year, count in year_counts.items():
    print(f"frequency for '{year}': {count}")

print("\nMonth Counts:")
for month, count in month_counts.items():
    print(f"frequency for months {month}: {count}")

print("\nDate Counts (Day Part Only):")
for day, count in date_counts.items():
    print(f"frequency for dates{day}: {count}")

**After attaining the information, I could target the average dates. Further process will be in section 3**

### **2.2: This block analyses data about stubs in further_extraction.ipynb.** 

2.21 First we look at the possibility that an article, that has the word "democracy" in its title, is a stub

2.211 Check the length of brief_data

In [ ]:
df_brief = pd.read_csv(brief_data)
len(df_brief)

2.212 Check how many rows there are in stub data

In [ ]:
df_stub = pd.read_csv(stub_data)
len(df_stub)

**349/1375 * 100% = 25.3818%.**  
**That means articles containing **"democracy**" in the title have 25.3818% to be stubs.**

2.22 I would like to see the frequency of countries appearing in my stub data.

2.221 Country names were attained by pasting all information from all_stubs_data.csv to IBMSPSS.

In [ ]:
country_names = ['South Korea', 'United Kingdom', 'Brazil', 'New Zealand', 'Democratic Republic of the Congo', 'Aruba', 'Kenya', 'Caribbean', 'Myanmar', 'Benin', 'Burundi', 'Åland', 'Algeria', 
'United States', 'China', 'Tonga', 'Turkey', 'France', 'Lesotho', 'Czechoslovakia', 'Senegal', 'Mali', 'Gabon', 'Philippines', 'Venezuela', 'Laos', 'Denmark', 'Djibouti', 'Iceland', 
'Guinea-Bissau', 'Kosovo', 'Portugal', 'Angola', 'Spain', 'Niger', 'Mayotte', 'Italy', 'Ecuador', 'Togo', 'Russia', 'Mauritania', 'Thailand', 'Luxembourg', 'Honduras', 'Gibraltar', 
'Yugoslavia', 'Knox County (Ohio, United States)', 'Guatemala', 'Iran', 'Peru', 'Cape Verde', 'Poland', 'Germany', 'Republic of the Congo']

2.222 Converting data format and iterations

In [ ]:
import re
import ast

data = df_stub['stubs']

country_pattern = r'\b(?:' + '|'.join(map(re.escape, country_names)) + r')\b'

country_name_counts = {}
for strings in data:
    stub_list = ast.literal_eval(strings) #convert data format
    for line in stub_list:
        matches = re.findall(country_pattern, line, flags=re.IGNORECASE)
        for match in matches:
            country_name_counts[match] = country_name_counts.get(match, 0) + 1

pprint(country_name_counts)


Now we have a list containing the frequencies of Countries appearing.  
**We could see from the list "country_name_counts" that "United States" occurs 20 times in my 349 rows of data, followed by "Benin" (17 times).**  
**This finding is very interesting and will be discussed further in Part 3 of this jupyter notebook about what it means.**

## Section 3 Analysis report

### 1st interesting finding: average last editing time

There are **1380 rows** of data in simplified_time. We could verify by using len()

In [ ]:
len(df_time)

Thus, the median will be the **NO.690** data inside the list.  
By processing the information attained in section 2.13: 

2023: [1019], 2022: [190], 2021: [67], 2020: [66], 2019: [16], 2018: [8]  
**Median of year(The NO.690 data) falls to '2023'**

January: [51], February: [73], March: [89], April: [79], May: [59], June: [74], July: [80], August: [137], September: [208], October: [310], November: [112], December: [94]  
**Median of Month(The NO.690 data) falls to 'September'**

01: [52], 02: [54], 03: [63], 04: [32], 05: [29], 06: [32], 07: [40], 08: [33], 09: [60], 10: [37], 11: [33], 12: [51], 13: [29], 14: [45], 15: [79], 16: [39], 17: [28], 18: [36], 19: [47], 20: [43], 21: [43], 22: [42], 23: [52], 24: [40], 25: [45], 26: [40], 27: [62], 28: [50], 29: [48], 30: [55], 31: [27].  
**Median of Date(The NO.690 data) falls to '16'**

The average last editing date for all 1380 pieces of Wikipedia data that contain "democracy" inside their titles is **16, Sept. 2023****

From the introduction of Wikipedia's official site: https://en.wikipedia.org/wiki/Wikipedia  
There are more than **61 million articles stored in Wikipedia, with 15 million edits per month**  
The expected duration for an article to be edited is approximately **4 Months**  
However, for our 1300+ data, the average edit time is less than **2.5 Months**, suggesting that the **democracy** topic is relatively more popular than most of the topics.

### 2nd interesting finding

**Stubs** are short articles that need further information to make the article complete.  
For our 1300+ articles about democracy, there is **25.3818%** that one article is a stub.

### 3rd interesting finding

Besides, for all 300+ **stubs**, the **United States** and **Benin**** are the 2 most frequent terms.  

For **United States**  
One possibility is that democracy in the United States is relatively more popular, also taking concerns of the political effect that US has to the world.  
Thus, more new concepts about "democracy" will emerge at a relatively fast speed. When they lack definitions, they become stubs.  

For **Benin**  
As for Benin, Benin is relatively less famous than the US. The reason why it has that many stubs may be due to a lack of enough information about this country, and its democracies.